In [1]:
import pickle
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import networkx as nx
import pyproj
import geopandas as gpd
import shapely
import pandas as pd
from shapely import Polygon, Point, LineString

from shapely.ops import transform
from pyproj import Transformer
from sklearn.cluster import KMeans

from tqdm.auto import tqdm

from pathlib import Path

In [2]:
src_dir = Path('~').expanduser() / 'data/d-osp/gtsm'

In [3]:
ds = xr.open_dataset(src_dir / 'filtered-currents.nc')
# ds = ds.isel(time=0)

In [4]:
ds_0 = ds.isel(time=0)
node_geoms = gpd.points_from_xy(ds_0['Mesh_face_x'], ds_0['Mesh_face_y'])
node_gdf = gpd.GeoDataFrame({'currents_u': ds_0['currents_u'], 'currents_v': ds_0['currents_v'], 'geometry': node_geoms})
node_gdf

,currents_u,currents_v,geometry
0,0.000,0.000,POINT (7.33154 53.31299)
1,0.000,0.000,POINT (7.33154 53.29834)
2,0.000,-0.002,POINT (7.34619 53.31299)
3,0.000,-0.002,POINT (7.34619 53.29834)
4,0.000,0.000,POINT (7.36084 53.31299)
...,...,...,...
19596,0.139,0.194,POINT (1.65527 51.22559)
19597,0.127,0.268,POINT (1.60669 51.22611)
19598,0.116,0.220,POINT (1.62598 51.22559)
19599,0.215,0.243,POINT (1.65527 51.19629)


In [5]:
def get_neighbors(ds, face_id):
    Mesh_face_nodes = ds['Mesh_face_nodes'].values - 1
    face_nodes = Mesh_face_nodes[face_id, :]
    face_nodes = face_nodes[~np.isnan(face_nodes)]
    face_nodes = face_nodes[face_nodes != -2]

    # Build node-to-face mapping
    node_to_faces = {}
    for f_id, nodes in enumerate(Mesh_face_nodes):
        valid_nodes = nodes[~np.isnan(nodes)]
        for node in valid_nodes:
            node_to_faces.setdefault(node, set()).add(f_id)

    # Collect neighbors
    neighbors = set()
    for node in face_nodes:
        neighbors.update(node_to_faces[node])

    neighbors.discard(face_id)
    return sorted(neighbors)

In [6]:
def build_edge_gdf(node_gdf):
    edge_gdf = {'source': [], 'target': [], 'geometry': []}
    for idx, node in tqdm(node_gdf.iterrows(), total=len(node_gdf)):
        source_node = idx
        target_nodes = get_neighbors(ds, source_node)
        for target_node in target_nodes:
            if target_node in edge_gdf['source']:
                continue
            geometry = LineString([node_gdf.iloc[source_node]['geometry'], node_gdf.iloc[target_node]['geometry']])
            
            edge_gdf['source'].append(source_node)
            edge_gdf['target'].append(target_node)
            edge_gdf['geometry'].append(geometry)

    edge_gdf = gpd.GeoDataFrame(edge_gdf)
    edge_gdf['edge_id'] = np.arange(len(edge_gdf))
    edge_gdf = edge_gdf.set_crs('EPSG:4326')
    edge_gdf = edge_gdf.to_crs('EPSG:3035')
    edge_gdf['length_m'] = shapely.length(edge_gdf['geometry'])
    return edge_gdf

In [7]:
edge_gdf_static = build_edge_gdf(node_gdf)

  0%|          | 0/19601 [00:00<?, ?it/s]

In [9]:
edge_gdf_static.to_file(src_dir / 'static-gtsm-edges.gpkg')

In [10]:
def compute_edge_values(node_gdf, edges_static):
    # Extract node-based values
    cu = node_gdf["currents_u"].to_numpy()
    cv = node_gdf["currents_v"].to_numpy()

    # Build an edge dataframe with new values only
    df = pd.DataFrame({
        "edge_id": edges_static["edge_id"],
        "currents_u": (cu[edges_static["source"].values] + cu[edges_static["target"].values]) / 2,
        "currents_v": (cv[edges_static["source"].values] + cv[edges_static["target"].values]) / 2,
    })

    # Merge back into a GeoDataFrame
    return edges_static.merge(df, on="edge_id")

In [11]:
compute_edge_values(node_gdf, edge_gdf_static)

,source,target,geometry,edge_id,length_m,currents_u,currents_v
0,0,1,"LINESTRING (4143175.799 3359374.03, 4143115.23...",0,1630.283230,0.0000,0.0000
1,0,2,"LINESTRING (4143175.799 3359374.03, 4144151.45...",1,976.312819,0.0000,-0.0010
2,0,3,"LINESTRING (4143175.799 3359374.03, 4144091.22...",2,1900.030262,0.0000,-0.0010
3,1,2,"LINESTRING (4143115.232 3357744.872, 4144151.4...",3,1900.670496,0.0000,-0.0010
4,1,3,"LINESTRING (4143115.232 3357744.872, 4144091.2...",4,976.645355,0.0000,-0.0010
...,...,...,...,...,...,...,...
76944,19589,19590,"LINESTRING (3744883.247 3149911.569, 3743216.6...",76944,3852.417801,0.2940,0.2575
76945,19590,19595,"LINESTRING (3743216.643 3153384.832, 3741183.9...",76945,2045.884056,0.2490,0.2460
76946,19591,19592,"LINESTRING (3739891.873 3160333.092, 3741552.8...",76946,3851.029701,0.1395,0.1830
76947,19591,19596,"LINESTRING (3739891.873 3160333.092, 3739521.5...",76947,3262.589712,0.1225,0.1790


In [12]:
def create_nodes_and_edges_from_ds(ds, edge_gdf_static, timestep):
    ds_t = ds.isel(time=timestep)
    node_geoms = gpd.points_from_xy(ds_t['Mesh_face_x'], ds_t['Mesh_face_y'])
    node_gdf = gpd.GeoDataFrame({'currents_u': ds_t['currents_u'], 'currents_v': ds_t['currents_v'], 'geometry': node_geoms})

    edge_gdf = compute_edge_values(node_gdf=node_gdf, edges_static=edge_gdf_static)
    node_gdf = node_gdf.set_crs("EPSG:4326")
    node_gdf = node_gdf.to_crs("EPSG:3035")
    return node_gdf, edge_gdf

In [13]:
create_nodes_and_edges_from_ds(ds, edge_gdf_static, timestep=20)[1]

,source,target,geometry,edge_id,length_m,currents_u,currents_v
0,0,1,"LINESTRING (4143175.799 3359374.03, 4143115.23...",0,1630.283230,-0.0005,-0.0070
1,0,2,"LINESTRING (4143175.799 3359374.03, 4144151.45...",1,976.312819,-0.0020,-0.0060
2,0,3,"LINESTRING (4143175.799 3359374.03, 4144091.22...",2,1900.030262,0.0000,-0.0060
3,1,2,"LINESTRING (4143115.232 3357744.872, 4144151.4...",3,1900.670496,-0.0005,-0.0060
4,1,3,"LINESTRING (4143115.232 3357744.872, 4144091.2...",4,976.645355,0.0015,-0.0060
...,...,...,...,...,...,...,...
76944,19589,19590,"LINESTRING (3744883.247 3149911.569, 3743216.6...",76944,3852.417801,-0.4450,-0.4640
76945,19590,19595,"LINESTRING (3743216.643 3153384.832, 3741183.9...",76945,2045.884056,-0.3955,-0.4680
76946,19591,19592,"LINESTRING (3739891.873 3160333.092, 3741552.8...",76946,3851.029701,-0.2620,-0.4585
76947,19591,19596,"LINESTRING (3739891.873 3160333.092, 3739521.5...",76947,3262.589712,-0.2485,-0.4790


In [14]:
def create_graph(node_gdf, edge_gdf):
    G = nx.from_pandas_edgelist(edge_gdf, edge_attr=True)
    node_attributes = node_gdf.to_dict('index')
    nx.set_node_attributes(G, node_attributes)

    src_crs = "EPSG:3035"
    dst_crs = "EPSG:4326"

    # Build a transformer. always_xy=True ensures (lon, lat) order.
    project = Transformer.from_crs(src_crs, dst_crs, always_xy=True).transform
    G = add_node_xy(G)
    # G = add_edge_geometries(G)
    # G = add_node_geometries(G, project=project)
    return G

def create_coarse_graph(node_gdf, edge_gdf):
    G = nx.from_pandas_edgelist(edge_gdf, edge_attr=True)
    node_attributes = node_gdf.to_dict('index')
    nx.set_node_attributes(G, node_attributes)

    G2 = coarsen_mesh_spatial(G, 500)

    src_crs = "EPSG:3035"
    dst_crs = "EPSG:4326"

    # Build a transformer. always_xy=True ensures (lon, lat) order.
    project = Transformer.from_crs(src_crs, dst_crs, always_xy=True).transform
    G2 = add_edge_geometries(G2)
    G2 = add_node_geometries(G2, project=project)
    return G2

def add_node_xy(G):
    for u in G.nodes():
        x, y = G.nodes[u]['geometry'].x, G.nodes[u]['geometry'].y
        G.nodes[u]['x'] = x
        G.nodes[u]['y'] = y
    return G

def add_edge_geometries(G):
    for u, v in G.edges():
        x1, y1 = G.nodes[u]['x'], G.nodes[u]['y']
        x2, y2 = G.nodes[v]['x'], G.nodes[v]['y']
        G.edges[u, v]['geometry'] = LineString([(x1, y1), (x2, y2)])
        G.edges[u, v]['length'] = shapely.length(G.edges[u, v]['geometry'])
    return G

def add_node_geometries(G, project):
    for u in G.nodes():
        x, y = G.nodes[u]['x'], G.nodes[u]['y']
        geometry = shapely.Point([x, y])
        geometry = transform(project, geometry)

        G.nodes[u]['geometry'] = geometry

    return G


def graph_edges_to_gdf(G):
    edge_data = []
    for u, v, data in G.edges(data=True):
        # Get coordinates of the two super-nodes
        x1, y1 = G.nodes[u]['x'], G.nodes[u]['y']
        x2, y2 = G.nodes[v]['x'], G.nodes[v]['y']
        # Create LineString geometry
        geom = LineString([(x1, y1), (x2, y2)])
        # Collect attributes
        edge_data.append({
            'source': u,
            'target': v,
            'currents_u': data.get('currents_u'),
            'currents_v': data.get('currents_v'),
            'length': data.get('length'),
            # 'edge_id': data.get('edge_id'),
            'geometry': geom
        })
    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(edge_data, geometry='geometry', crs="EPSG:3035")  # or your CRS
    return gdf

def compute_direction(p1, p2):
    x1, y1 = p1
    x2, y2 = p2

    # Direction unit vector
    dx, dy = x2 - x1, y2 - y1
    length = np.sqrt(dx**2 + dy**2)
    if length == 0:
        direction = 0
    else:
        direction = (dx/length, dy/length)

    return direction


def coarsen_mesh_spatial(G, target_clusters):
    # Extract coordinates
    coords = np.array([[G.nodes[n]['geometry'].x, G.nodes[n]['geometry'].y] for n in G.nodes()])
    node_ids = list(G.nodes())

    # Cluster nodes using KMeans
    kmeans = KMeans(n_clusters=target_clusters, random_state=42)
    labels = kmeans.fit_predict(coords)

    # Create new graph
    H = nx.Graph()

    # Add super-nodes with averaged coordinates
    for cluster_id in range(target_clusters):
        cluster_nodes = [node_ids[i] for i in range(len(node_ids)) if labels[i] == cluster_id]
        avg_x = np.mean([G.nodes[n]['geometry'].x for n in cluster_nodes])
        avg_y = np.mean([G.nodes[n]['geometry'].y for n in cluster_nodes])
        H.add_node(cluster_id, x=avg_x, y=avg_y, members=cluster_nodes)

    # Build edges between clusters and average attributes
    edge_dict = {}
    for u, v, data in G.edges(data=True):
        cu = labels[node_ids.index(u)]
        cv = labels[node_ids.index(v)]
        if cu != cv:
            key = tuple(sorted((cu, cv)))
            if key not in edge_dict:
                edge_dict[key] = {'currents_u': [], 'currents_v': []}
            edge_dict[key]['currents_u'].append(data['currents_u'])
            edge_dict[key]['currents_v'].append(data['currents_v'])

    # Add averaged edges to new graph
    for (cu, cv), attrs in edge_dict.items():
        avg_currents_u = np.mean(attrs['currents_u'])
        avg_currents_v = np.mean(attrs['currents_v'])
        H.add_edge(cu, cv, currents_u=avg_currents_u, currents_v=avg_currents_v)

    return H

def create_digraph_from_clusters(G):
    src_crs = "EPSG:3035"
    dst_crs = "EPSG:4326"

    # Build a transformer. always_xy=True ensures (lon, lat) order.
    project = Transformer.from_crs(src_crs, dst_crs, always_xy=True).transform
    G2 = nx.DiGraph()
    for u, v, data in G.edges(data=True):
        # Node positions
        x1, y1 = G.nodes[u]['x'], G.nodes[u]['y']
        x2, y2 = G.nodes[v]['x'], G.nodes[v]['y']

        direction = compute_direction((x1, y1), (x2, y2))
        direction_u, direction_v = direction

        data['direction_u'] = direction_u
        data['direction_v'] = direction_v

        if u not in G2.nodes:
            G2.add_node(u, **G.nodes[u])
        if v not in G2.nodes:
            G2.add_node(v, **G.nodes[v])

        current_u = data['currents_u']
        current_v = data['currents_v']

        current = np.dot([current_u, current_v], [direction_u, direction_v])
        data['Info'] = {'Current': current}

        geometry = data['geometry']
        geometry = transform(project, geometry)
        data['geometry'] = geometry

        G2.add_edge(u, v, **data)

    for v, u, data in G.edges(data=True):
        # Node positions
        x1, y1 = G.nodes[u]['x'], G.nodes[u]['y']
        x2, y2 = G.nodes[v]['x'], G.nodes[v]['y']
            
        direction = compute_direction((x1, y1), (x2, y2))
        direction_u, direction_v = direction

        data['direction_u'] = direction_u
        data['direction_v'] = direction_v

        current_u = data['currents_u']
        current_v = data['currents_v']

        current = np.dot([current_u, current_v], [direction_u, direction_v])
        data['Info'] = {'Current': current}

        geometry = data['geometry']
        geometry = transform(project, geometry)
        data['geometry'] = geometry

        G2.add_edge(u, v, **data)

    return G2

In [15]:
def create_gtsm_digraph(ds, edge_gdf_static, timestep):
    node_gdf, edge_gdf = create_nodes_and_edges_from_ds(ds=ds, edge_gdf_static=edge_gdf_static, timestep=timestep)
    G = create_coarse_graph(node_gdf=node_gdf, edge_gdf=edge_gdf)
    G_d = create_digraph_from_clusters(G)
    return G_d

In [16]:
G = create_gtsm_digraph(ds=ds, edge_gdf_static=edge_gdf_static, timestep=20)

In [17]:
graph_edges_to_gdf(G)

,source,target,currents_u,currents_v,length,geometry
0,0,246,0.351600,-0.069825,9808.474510,"LINESTRING (3980190.972 3349209.844, 3986936.4..."
1,0,72,0.186182,-0.070682,11942.122459,"LINESTRING (3980190.972 3349209.844, 3991709.1..."
2,0,322,0.299417,0.218500,12409.823797,"LINESTRING (3980190.972 3349209.844, 3986902.6..."
3,0,432,0.163022,0.083478,10257.500061,"LINESTRING (3980190.972 3349209.844, 3978265.5..."
4,0,92,0.394176,-0.156529,11254.422291,"LINESTRING (3980190.972 3349209.844, 3978759.5..."
...,...,...,...,...,...,...
2729,173,493,0.412118,0.333500,11797.261387,"LINESTRING (3965870.781 3375768.392, 3957493.6..."
2730,189,288,-0.850444,-0.268111,11121.063949,"LINESTRING (3913599.896 3183763.344, 3903901.7..."
2731,295,386,0.043250,-0.071781,9466.600347,"LINESTRING (4111595.541 3515917.56, 4107955.76..."
2732,295,494,0.045132,-0.071658,9265.780763,"LINESTRING (4111595.541 3515917.56, 4112230.58..."


In [36]:
G.edges

EdgeView([(0, 246), (0, 72), (0, 322), (0, 432), (0, 92), (0, 103), (0, 334), (1, 426), (1, 232), (1, 375), (1, 454), (1, 279), (2, 280), (2, 165), (2, 164), (2, 407), (2, 323), (2, 150), (2, 302), (3, 488), (3, 465), (3, 439), (3, 448), (3, 123), (4, 392), (4, 339), (4, 262), (4, 67), (4, 361), (4, 408), (5, 400), (5, 357), (5, 293), (5, 307), (6, 147), (6, 485), (6, 473), (6, 242), (7, 288), (7, 367), (8, 364), (8, 450), (8, 227), (8, 191), (8, 477), (9, 459), (9, 224), (9, 190), (9, 430), (10, 241), (10, 270), (10, 143), (10, 186), (10, 326), (10, 104), (10, 221), (11, 376), (11, 134), (11, 109), (11, 306), (11, 304), (11, 87), (12, 145), (12, 458), (12, 215), (12, 250), (12, 437), (12, 164), (13, 205), (13, 222), (13, 358), (13, 405), (13, 180), (14, 341), (14, 493), (14, 154), (14, 350), (14, 413), (14, 391), (15, 393), (15, 389), (15, 171), (16, 272), (16, 354), (16, 449), (16, 140), (16, 282), (17, 182), (17, 320), (17, 471), (17, 124), (17, 497), (17, 294), (17, 176), (18, 476)

In [44]:
edge_gdf

,source,target,geometry,edge_id,length,currents_u,currents_v
0,0,1,"LINESTRING (4143175.799 3359374.03, 4143115.23...",0,1630.283230,0.0000,0.0000
1,0,2,"LINESTRING (4143175.799 3359374.03, 4144151.45...",1,976.312819,0.0000,-0.0010
2,0,3,"LINESTRING (4143175.799 3359374.03, 4144091.22...",2,1900.030262,0.0000,-0.0010
3,1,2,"LINESTRING (4143115.232 3357744.872, 4144151.4...",3,1900.670496,0.0000,-0.0010
4,1,3,"LINESTRING (4143115.232 3357744.872, 4144091.2...",4,976.645355,0.0000,-0.0010
...,...,...,...,...,...,...,...
76944,19589,19590,"LINESTRING (3744883.247 3149911.569, 3743216.6...",76944,3852.417801,0.2940,0.2575
76945,19590,19595,"LINESTRING (3743216.643 3153384.832, 3741183.9...",76945,2045.884056,0.2490,0.2460
76946,19591,19592,"LINESTRING (3739891.873 3160333.092, 3741552.8...",76946,3851.029701,0.1395,0.1830
76947,19591,19596,"LINESTRING (3739891.873 3160333.092, 3739521.5...",76947,3262.589712,0.1225,0.1790


In [50]:
edge_gdf_static_c

,source,target,currents_u,currents_v,length,edge_id,geometry
0,0,246,-0.459000,-0.038475,9808.474510,0,"LINESTRING (3980190.972 3349209.844, 3986936.4..."
1,0,72,-0.258545,-0.017773,11942.122459,1,"LINESTRING (3980190.972 3349209.844, 3991709.1..."
2,0,322,-0.348583,-0.319167,12409.823797,2,"LINESTRING (3980190.972 3349209.844, 3986902.6..."
3,0,432,-0.188000,-0.203630,10257.500061,3,"LINESTRING (3980190.972 3349209.844, 3978265.5..."
4,0,92,-0.656088,0.267853,11254.422291,4,"LINESTRING (3980190.972 3349209.844, 3978759.5..."
...,...,...,...,...,...,...,...
1362,465,488,-0.009417,-0.090667,15434.159450,1362,"LINESTRING (3908582.009 3429422.202, 3921305.7..."
1363,467,497,-0.186556,-0.492556,12060.339048,1363,"LINESTRING (3941779.829 3291818.658, 3933558.4..."
1364,470,490,-0.232375,-0.480125,15222.965287,1364,"LINESTRING (3885015.114 3263959.754, 3870461.8..."
1365,478,482,0.031417,0.015083,17308.668981,1365,"LINESTRING (3927241.568 3484444.031, 3937424.9..."
